# 02 — Preparação dos dados
Continuação da EDA: validação do CSV, divisão por utilizador, seleção explícita de entradas e preparação ajustada apenas ao treino.

In [1]:
from pathlib import Path
import sys
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/cybercrime_forensic_dataset.csv').exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.dados import carregar_dados, dividir_dados, resumo_divisao
df = carregar_dados()
print("Dimensão:", df.shape)
df.info()
display(df.isna().sum().to_frame("Ausentes"))
display(df.nunique().to_frame("Valores distintos"))
print("Duplicados completos:", df.duplicated().sum())
display(df["Label"].value_counts())

Dimensão: (7400, 11)
<class 'pandas.DataFrame'>
RangeIndex: 7400 entries, 0 to 7399
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Timestamp          7400 non-null   str    
 1   User_ID            7400 non-null   int64  
 2   IP_Address         7400 non-null   str    
 3   Activity_Type      7400 non-null   str    
 4   Resource_Accessed  7400 non-null   str    
 5   File_Name          3196 non-null   str    
 6   Action             7400 non-null   str    
 7   Login_Attempts     2105 non-null   float64
 8   File_Size          3196 non-null   float64
 9   Anomaly_Type       7400 non-null   str    
 10  Label              7400 non-null   str    
dtypes: float64(2), int64(1), str(8)
memory usage: 636.1 KB


,Ausentes
Timestamp,0
User_ID,0
IP_Address,0
Activity_Type,0
Resource_Accessed,0
File_Name,4204
Action,0
Login_Attempts,5295
File_Size,4204
Anomaly_Type,0


,Valores distintos
Timestamp,7400
User_ID,5031
IP_Address,7353
Activity_Type,7
Resource_Accessed,36
File_Name,6
Action,7
Login_Attempts,10
File_Size,2382
Anomaly_Type,5


Duplicados completos: 0


Label
Normal        6167
Suspicious    1233
Name: count, dtype: int64

## Divisão por utilizador
Existem vários registos do mesmo utilizador. O objetivo experimental fica definido como classificar atividades de **utilizadores não observados no treino**, dentro da população e período simulados. GroupShuffleSplit, com semente 42, reserva 20% dos utilizadores para teste; dos restantes, 25% ficam para validação (aproximadamente 60%/20%/20% dos utilizadores no total).
As percentagens de linhas e classes podem diferir: não se estratificam os rótulos nem se procura uma divisão com melhores resultados. A semente não será ajustada em função das métricas.
Uma divisão por linhas permitiria utilizadores repetidos nos conjuntos; uma divisão temporal responderia à previsão de períodos futuros, que não é a alegação desta experiência. Não existe um identificador de incidente para garantir a separação de acontecimentos relacionados.
O teste já foi abrangido pela EDA original; essa limitação é reconhecida. As contagens seguintes verificam apenas a composição e não serão utilizadas para selecionar modelos.

In [2]:
conjuntos = dividir_dados(df)
display(resumo_divisao(conjuntos))
for a, b in [("treino", "validacao"), ("treino", "teste"), ("validacao", "teste")]:
    assert set(conjuntos[a]["User_ID"]).isdisjoint(conjuntos[b]["User_ID"])
    assert set(conjuntos[a].index).isdisjoint(conjuntos[b].index)
assert sum(len(parte) for parte in conjuntos.values()) == len(df)
print("Divisão congelada: sem sobreposição de linhas ou utilizadores.")

,Registos,Utilizadores,Normal,Suspicious,% Suspicious
Conjunto,,,,,
treino,4416,3018,3682,734,16.62
validacao,1478,1006,1237,241,16.31
teste,1506,1007,1248,258,17.13


Divisão congelada: sem sobreposição de linhas ou utilizadores.


## Decisões sobre as características
- User_ID fica apenas como grupo de divisão, não como entrada. A separação por utilizador impede memorizar a sua identidade entre treino e avaliação.
- IP_Address é quase único, sem topologia ou contexto de rede confirmado; exclui-se para evitar memorizar endereços sem significado transferível.
- Resource_Accessed é conservado como categoria. File_Name coincide com o último componente do recurso nos registos em que existe; exclui-se a coluna redundante, sem perder essa informação.
- Timestamp fornece seno e cosseno da hora e do dia da semana: representam a continuidade dos ciclos sem usar o instante único como identificador.
- Activity_Type, Action, Login_Attempts e File_Size são conservados. A previsão refere-se a uma atividade já registada, não à antecipação do seu resultado.
- Label, Label_bin e Anomaly_Type são explicitamente proibidos nas entradas.
## Ausências, escala e categorias
Os números estruturalmente não aplicáveis tornam-se zero; um número desconhecido numa atividade aplicável é recusado para não o confundir com ausência estrutural. File_Name não precisa de imputação por ter sido excluído. As categorias restantes ausentes usam Unknown e categorias novas são aceites pelo OneHotEncoder sem reaprender categorias.
A escala e a codificação são ajustadas exclusivamente no treino e, posteriormente, dentro de cada fold. O StandardScaler é relevante para regressão logística, KNN e SVM; a escala comum não é necessária às árvores, mas mantém a representação comparável. Não são eliminadas linhas por ausência estrutural nem por um critério automático de outliers. Na EDA, não foram identificados valores fora de 1,5 × IQR; anomalias de contexto permanecem possíveis.

In [3]:
import numpy as np
from src.preparacao import selecionar_entradas, obter_alvo, criar_preparacao, PROIBIDAS
X_treino = selecionar_entradas(conjuntos["treino"])
X_validacao = selecionar_entradas(conjuntos["validacao"])
y_treino = obter_alvo(conjuntos["treino"])
assert not (PROIBIDAS & set(X_treino.columns))
preparacao = criar_preparacao()
matriz_treino = preparacao.fit_transform(X_treino, y_treino)
matriz_validacao = preparacao.transform(X_validacao)
assert np.isfinite(matriz_treino).all() and np.isfinite(matriz_validacao).all()
print("Entradas permitidas:", list(X_treino.columns))
print("Matrizes treino/validação:", matriz_treino.shape, matriz_validacao.shape)
print("Número de características transformadas:", len(preparacao.get_feature_names_out()))
print("O teste não foi utilizado para ajustar ou comparar transformações.")

Entradas permitidas: ['Timestamp', 'Activity_Type', 'Resource_Accessed', 'Action', 'Login_Attempts', 'File_Size']
Matrizes treino/validação: (4416, 56) (1478, 56)
Número de características transformadas: 56
O teste não foi utilizado para ajustar ou comparar transformações.


## Desequilíbrio
A distribuição natural é preservada em todos os conjuntos. Serão comparadas referências simples e, nos algoritmos que o suportam, ponderação das classes. Não será aplicada reamostragem global. Precision, Recall, F1 e Average Precision complementam a accuracy.
**Próxima etapa:** referências e comparação no notebook 03. A preparação comum integra cada Pipeline de treino para impedir ajuste fora dos folds.